<div style='text-align: center; padding: 30px; background: linear-gradient(135deg, #0f0c29 0%, #302b63 50%, #24243e 100%); border-radius: 15px; margin: 10px 0; box-shadow: 0 10px 30px rgba(0,0,0,0.3);'>
  <h1 style='color: #ffffff; margin: 0 0 8px 0; font-size: 2.5em;'>🏃 ARDY Motion Generator</h1>
  <h3 style='color: #c0c0ff; margin: 0 0 5px 0; font-weight: 400;'>Kaggle Dual T4 Edition</h3>
  <p style='color: #aaa; margin: 0; text-align: center;'>Text-to-Motion • Interactive Demo • NVIDIA ARDY + LLM2Vec (Llama-3-8B)</p>
</div>

<p align="center">
  <a href="https://www.youtube.com/@thebuildai?sub_confirmation=1"><img src="https://img.shields.io/badge/YouTube-SUBSCRIBE-red?style=for-the-badge&logo=youtube&logoColor=white" /></a>
  <a href="https://www.instagram.com/thebuildai/"><img src="https://img.shields.io/badge/Instagram-FOLLOW-E4405F?style=for-the-badge&logo=instagram&logoColor=white" /></a>
  <a href="https://www.tiktok.com/@the.build.ai"><img src="https://img.shields.io/badge/TikTok-FOLLOW-000000?style=for-the-badge&logo=tiktok&logoColor=white" /></a>
  <a href="https://github.com/cafermutluozkan"><img src="https://img.shields.io/badge/GitHub-FOLLOW-181717?style=for-the-badge&logo=github&logoColor=white" /></a>
</p>

---

### 🚀 What is this notebook?

Interactive **Text-to-Motion** generation using **NVIDIA ARDY** (SIGGRAPH 2026) with a task-level dual-GPU layout: the motion diffusion model runs on **GPU 0**, the Llama-3-8B text encoder runs as a local API service on **GPU 1** (with automatic CPU fallback).

| Feature | Detail |
|---|---|
| **Model** | NVIDIA ARDY Core (Bones rig, 20 FPS, Horizon 8/40) |
| **GPU** | T4 x2 (2 × 16 GB VRAM) |
| **Text encoder** | LLM2Vec (Meta-Llama-3-8B-Instruct), local Gradio API on port 9550 |
| **Output** | `.npz` motion (posed joints, rotations, foot contacts) |
| **Extras** | T4 FP16 patch, auto CPU fallback on OOM, Cloudflare-tunneled interactive Viser demo |

### ⚡ Quick Start
1. **Settings → Accelerator → GPU T4 x2** and turn on **Internet**
2. Request access to `meta-llama/Meta-Llama-3-8B-Instruct` on Hugging Face, then add a Kaggle secret named `HF_TOKEN`
3. **Runtime → Run All** — after the install cell finishes, do one **Restart & Run All** (dependency pins require a fresh kernel)
4. Download the `.npz` result, or open the interactive demo URL from Step 10

---

## Step 1: Verify the Kaggle runtime

In [ ]:
import os
import platform
import subprocess
import sys

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

print("Python:", sys.version)
print("Platform:", platform.platform())
subprocess.run(["nvidia-smi"], check=True)

import torch

print("PyTorch:", torch.__version__)
print("CUDA runtime:", torch.version.cuda)
print("GPU count:", torch.cuda.device_count())
for index in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(index)
    print(
        f"cuda:{index}: {props.name}, "
        f"{props.total_memory / 1024**3:.1f} GiB, "
        f"compute capability {props.major}.{props.minor}"
    )

assert torch.cuda.device_count() >= 2, (
    "This notebook requires Kaggle's GPU T4 x2 accelerator. "
    "Open Notebook options and select GPU T4 x2."
)

---

## Step 2: Read the Hugging Face token

The token is first read from Kaggle Secrets and is never printed. If `HF_TOKEN` is not attached, the cell securely asks for it with a masked input as a session-only fallback. Your Hugging Face account must already have access to Meta Llama 3 8B Instruct.

In [ ]:
from getpass import getpass
from kaggle_secrets import UserSecretsClient

token_source = "Kaggle Secrets"
try:
    hf_token = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    token_source = "masked session input"
    print(
        "HF_TOKEN is not attached to this notebook. Enter it below; "
        "the value will be masked and kept only for this runtime."
    )
    hf_token = getpass("Hugging Face token: ").strip()

if not hf_token:
    raise RuntimeError("No Hugging Face token was provided.")

os.environ["HF_TOKEN"] = hf_token
os.environ["HUGGING_FACE_HUB_TOKEN"] = hf_token
print(f"HF_TOKEN loaded from {token_source}.")

---

## Step 3: Clone ARDY & install dependencies

The notebook deliberately installs the `demo` extras, not the TensorRT extras. Plain PyTorch is easier to validate first on Kaggle and avoids spending time building TensorRT engines.

In [ ]:
from pathlib import Path

WORKDIR = Path("/kaggle/working")
REPO_DIR = WORKDIR / "ardy"

if not REPO_DIR.exists():
    subprocess.run(
        ["git", "clone", "--depth", "1", "https://github.com/nv-tlabs/ardy.git", str(REPO_DIR)],
        check=True,
    )
else:
    print(f"Using existing checkout: {REPO_DIR}")

subprocess.run(
    [sys.executable, "-m", "pip", "install", "--upgrade", "pip"],
    check=True,
)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-e", ".[demo]"],
    cwd=REPO_DIR,
    check=True,
)
print("ARDY installation completed.")

# Kaggle ships torchao 0.10.0, which peft>=0.19 rejects ("only versions above
# 0.16.0 are supported") while merging the LLM2Vec PEFT adapter -- the text
# encoder server then exits with code 1. ARDY does not use torchao, so remove
# it (upgrading torchao would risk a torch version conflict instead).
subprocess.run(
    [sys.executable, "-m", "pip", "uninstall", "-y", "torchao"],
    check=False,
)
print("Removed Kaggle's incompatible torchao (if present).")


---

## Step 4: Apply the T4 FP16 compatibility patch

Upstream ARDY defaults to BF16 for LLM2Vec. NVIDIA T4 is SM 7.5 and supports FP16 Tensor Cores but not native BF16. This idempotent patch keeps BF16 on Ampere/Ada/Hopper GPUs and selects FP16 only on pre-Ampere CUDA devices.

In [ ]:
LOAD_MODEL_FILE = REPO_DIR / "ardy" / "model" / "load_model.py"
source = LOAD_MODEL_FILE.read_text(encoding="utf-8")
old = """    dtype = torch.float32 if fp32 else torch.bfloat16
    return text_encoder.to(device=device, dtype=dtype)"""
new = """    if fp32:
        dtype = torch.float32
    elif str(device).startswith(\"cuda\") and torch.cuda.get_device_capability(device)[0] < 8:
        # Turing/T4 has FP16 Tensor Cores but no native BF16 support.
        print(f\"Device {device} has no native BF16 support; using float16.\")
        dtype = torch.float16
    else:
        dtype = torch.bfloat16
    return text_encoder.to(device=device, dtype=dtype)"""

if new in source:
    print("FP16 compatibility patch is already present.")
elif old in source:
    LOAD_MODEL_FILE.write_text(source.replace(old, new, 1), encoding="utf-8")
    print("Applied T4 FP16 compatibility patch.")
else:
    raise RuntimeError(
        "ARDY's load_text_encoder implementation changed upstream; "
        "review load_model.py before applying the compatibility patch."
    )


# Gradio auto-creates a public share link on hosted notebooks and hides server
# errors from API clients. Pin share=False and show_error=True so failures
# surface directly in this notebook.
SERVER_FILE = REPO_DIR / "scripts" / "run_text_encoder_server.py"
server_source = SERVER_FILE.read_text(encoding="utf-8")
old_launch = "demo.launch(server_name=args.host, server_port=args.port)"
new_launch = (
    "demo.launch(server_name=args.host, server_port=args.port, "
    "share=False, show_error=True)"
)
if new_launch in server_source:
    print("Gradio launch patch is already present.")
elif old_launch in server_source:
    SERVER_FILE.write_text(
        server_source.replace(old_launch, new_launch, 1), encoding="utf-8"
    )
    print("Applied Gradio launch patch (share=False, show_error=True).")
else:
    raise RuntimeError(
        "run_text_encoder_server.py launch call changed upstream; "
        "review the file before applying the Gradio launch patch."
    )


---

## Step 5: Validate gated model access

This fails early with a clear error if Meta has not granted your account access or the token lacks read permission.

In [ ]:
from huggingface_hub import hf_hub_download, whoami

LLAMA_REPO = "meta-llama/Meta-Llama-3-8B-Instruct"
identity = whoami(token=hf_token)
print("Hugging Face account used by this token:", identity["name"])
print("Confirm that this is the account approved on the Llama model page.")
try:
    # Metadata is public even for gated repositories; downloading a real
    # file is required to prove that this account/token has gated access.
    config_path = hf_hub_download(
        repo_id=LLAMA_REPO, filename="config.json", token=hf_token
    )
except Exception as exc:
    raise RuntimeError(
        f"Cannot download {LLAMA_REPO}/config.json. Open the model page, "
        "request/accept Meta's access terms, wait for approval, and use a token "
        "from that same approved account with gated-repository read access."
    ) from exc
print("Gated model file access confirmed:", config_path)

---

## Step 6: Start the text encoder service (GPU 1, CPU fallback)

The first start downloads roughly 15–20 GB of Llama/LLM2Vec files and can take several minutes. The process remains alive in the background while subsequent cells run.

In [ ]:
import socket
import time

SERVER_LOG = WORKDIR / "ardy_text_encoder.log"
SERVER_PID = WORKDIR / "ardy_text_encoder.pid"

# Llama-3-8B in fp16 needs ~16 GB, which is at (or just above) what a single
# T4 offers. If the GPU runs out of memory this cell automatically retries the
# encoder on CPU; encoding is slower but only runs once per prompt.
#
# CUDA_VISIBLE_DEVICES="1" makes the server see only the physical GPU 1 (as
# "cuda:0"). This matters even on the CPU fallback: LLM2Vec switches to
# multi-process encoding whenever it detects 2+ GPUs, and shipping the model
# through torch shared memory crashes on Kaggle's tiny /dev/shm
# ("unable to allocate shared memory"). With a single visible device it
# always takes the single-process path.
TEXT_ENCODER_DEVICE = "cuda:0"  # physical GPU 1 via CUDA_VISIBLE_DEVICES below


def launch_text_encoder(device):
    server_env = os.environ.copy()
    server_env["TEXT_ENCODER_DEVICE"] = device
    server_env["HF_ENABLE_PARALLEL_LOADING"] = "YES"
    server_env["CUDA_VISIBLE_DEVICES"] = "1"  # single visible device, see note above
    log_handle = SERVER_LOG.open("w", encoding="utf-8")
    process = subprocess.Popen(
        [
            sys.executable,
            "scripts/run_text_encoder_server.py",
            "--host",
            "127.0.0.1",
            "--port",
            "9550",
            "--device",
            device,
        ],
        cwd=REPO_DIR,
        env=server_env,
        stdout=log_handle,
        stderr=subprocess.STDOUT,
    )
    return process, log_handle


def port_is_open(host="127.0.0.1", port=9550):
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as sock:
        sock.settimeout(1)
        return sock.connect_ex((host, port)) == 0


# Stop a server previously launched by this notebook, if one exists.
if SERVER_PID.exists():
    try:
        old_pid = int(SERVER_PID.read_text().strip())
        os.kill(old_pid, 15)
        time.sleep(2)
    except (ProcessLookupError, ValueError):
        pass

device = TEXT_ENCODER_DEVICE
attempted_cpu_fallback = False

while True:
    text_encoder_process, log_handle = launch_text_encoder(device)
    SERVER_PID.write_text(str(text_encoder_process.pid))
    print(f"Text encoder PID: {text_encoder_process.pid} (device={device})")
    print("Log file:", SERVER_LOG)

    deadline = time.time() + 1200
    last_report = 0
    return_code = None
    exit_log = None
    while time.time() < deadline:
        return_code = text_encoder_process.poll()
        if return_code is not None:
            log_handle.close()
            exit_log = SERVER_LOG.read_text(encoding="utf-8", errors="replace")
            break
        if port_is_open():
            break
        if time.time() - last_report > 30:
            print("Still loading the text encoder...")
            last_report = time.time()
        time.sleep(5)

    if port_is_open():
        print("Text encoder service is listening on http://127.0.0.1:9550/")
        break
    if exit_log is None:
        raise TimeoutError(f"Text encoder did not start. Inspect {SERVER_LOG}.")
    if (
        not attempted_cpu_fallback
        and device != "cpu"
        and "out of memory" in exit_log.lower()
    ):
        attempted_cpu_fallback = True
        device = "cpu"
        print("GPU 1 ran out of memory; retrying the text encoder on CPU.")
        continue
    print(exit_log)
    raise RuntimeError(f"Text encoder exited with code {return_code}.")

log_handle.flush()


---

## Step 7: Verify the encoder API & GPU placement

In [ ]:
# Import after installation/patching so this kernel sees the editable checkout.
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

from ardy.model.text_encoder_api import TextEncoderAPI

encoder_client = TextEncoderAPI("http://127.0.0.1:9550/").to(
    device="cuda:0", dtype=torch.float16
)
embedding, lengths = encoder_client(["A person walks forward."])
print("Embedding shape:", tuple(embedding.shape))
print("Embedding lengths:", lengths)
print("Returned tensor:", embedding.device, embedding.dtype)
assert embedding.device.index == 0
assert embedding.shape[-1] == 4096
subprocess.run(["nvidia-smi"], check=True)

---

## Step 8: Generate motion on GPU 0

`core8` is used for the first smoke test because its short prediction horizon is a conservative starting point. Increase duration/history after confirming that the full pipeline works.

In [ ]:
PROMPT = "A person walks forward and waves with the right hand."
MODEL = "core8"
DURATION_SECONDS = 5.0
DIFFUSION_STEPS = 10
HISTORY_FRAMES = 16  # Must be a positive multiple of 4 for Core models.
OUTPUT_NAME = "kaggle_dual_t4_demo"

generation_env = os.environ.copy()
generation_env["TEXT_ENCODER_MODE"] = "api"
generation_env["TEXT_ENCODER_URL"] = "http://127.0.0.1:9550/"

command = [
    sys.executable,
    "scripts/generate.py",
    PROMPT,
    "--model",
    MODEL,
    "--duration",
    str(DURATION_SECONDS),
    "--diffusion_steps",
    str(DIFFUSION_STEPS),
    "--history_frames",
    str(HISTORY_FRAMES),
    "--seed",
    "0",
    "--no-postprocess",
    "--output",
    OUTPUT_NAME,
]
print("Running:", " ".join(command))
subprocess.run(command, cwd=REPO_DIR, env=generation_env, check=True)
subprocess.run(["nvidia-smi"], check=True)

---

## Step 9: Inspect & download the result

In [ ]:
import numpy as np
from IPython.display import FileLink, display

output_path = REPO_DIR / "outputs" / f"{OUTPUT_NAME}.npz"
if not output_path.exists():
    raise FileNotFoundError(output_path)

motion = np.load(output_path, allow_pickle=False)
print("Output:", output_path)
print("Size:", f"{output_path.stat().st_size / 1024**2:.2f} MiB")
print("Arrays:")
for key in motion.files:
    value = motion[key]
    print(f"  {key:24s} shape={value.shape!s:18s} dtype={value.dtype}")

download_path = WORKDIR / output_path.name
download_path.write_bytes(output_path.read_bytes())
display(FileLink(str(download_path)))

---

## Optional: Plot a generated frame

This is a lightweight sanity check, not the full Viser animation viewer.

In [ ]:
import matplotlib.pyplot as plt

joints = motion["posed_joints"]
frame_index = len(joints) // 2
frame = joints[frame_index]

fig = plt.figure(figsize=(7, 7))
ax = fig.add_subplot(111, projection="3d")
ax.scatter(frame[:, 0], frame[:, 2], frame[:, 1], s=35)
ax.set_title(f"ARDY generated pose — frame {frame_index}")
ax.set_xlabel("X")
ax.set_ylabel("Z")
ax.set_zlabel("Y")
ax.set_box_aspect((1, 1, 1))
plt.show()

---

## Step 10 (Optional): Interactive demo in your browser

The main flow of this notebook is CLI-based. If you want ARDY's real interactive UI (Viser viewer, timeline, online prompts, kinematic constraints), the cell below starts `scripts/run_demo.py --no-compile` on GPU 0 and exposes it through a free Cloudflare quick tunnel (no account needed, WebSocket-capable).

Requirements: the text encoder service from section 6 must still be running — the demo detects it automatically (`auto` mode probes `127.0.0.1:9550`), so the 16 GB encoder is never loaded twice.

The printed `*.trycloudflare.com` URL is public while the tunnel is alive; anyone with the link can open the demo. Stop both processes with the cleanup cell at the end when you are done.

> On T4 without TensorRT the interactive stream runs slower than real time — that is expected. Keep durations short for a usable experience.

In [ ]:
import re

DEMO_LOG = WORKDIR / "ardy_demo.log"
DEMO_PID = WORKDIR / "ardy_demo.pid"
TUNNEL_LOG = WORKDIR / "cloudflared.log"
CLOUDFLARED = WORKDIR / "cloudflared"

# Download cloudflared (single static binary; free quick tunnels, no account).
if not CLOUDFLARED.exists():
    subprocess.run(
        [
            "wget",
            "-q",
            "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
            "-O",
            str(CLOUDFLARED),
        ],
        check=True,
    )
    CLOUDFLARED.chmod(0o755)
    print("cloudflared downloaded.")

# Stop a demo previously launched by this notebook, if one exists.
if DEMO_PID.exists():
    try:
        os.kill(int(DEMO_PID.read_text().strip()), 15)
        time.sleep(2)
    except (ProcessLookupError, ValueError):
        pass

# Start the interactive demo on GPU 0. It detects the already-running encoder
# service at 127.0.0.1:9550 automatically (auto mode), so nothing is reloaded.
demo_log_handle = DEMO_LOG.open("w", encoding="utf-8")
demo_process = subprocess.Popen(
    [sys.executable, "scripts/run_demo.py", "--no-compile"],
    cwd=REPO_DIR,
    env=os.environ.copy(),
    stdout=demo_log_handle,
    stderr=subprocess.STDOUT,
)
DEMO_PID.write_text(str(demo_process.pid))
print("Demo PID:", demo_process.pid, "| log:", DEMO_LOG)


def _wait_for_port(port, process, log_path, deadline_s=600):
    deadline = time.time() + deadline_s
    while time.time() < deadline:
        rc = process.poll()
        if rc is not None:
            print(log_path.read_text(encoding="utf-8", errors="replace"))
            raise RuntimeError(f"Process exited with code {rc}; log above.")
        with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as sock:
            sock.settimeout(1)
            if sock.connect_ex(("127.0.0.1", port)) == 0:
                return True
        time.sleep(5)
    raise TimeoutError(f"Port {port} never opened; inspect {log_path}.")


_wait_for_port(2333, demo_process, DEMO_LOG)
print("Viser demo is up on 127.0.0.1:2333. Starting the tunnel...")

# Cloudflare quick tunnel exposes the demo over HTTPS (WebSocket-capable).
tunnel_log_handle = TUNNEL_LOG.open("w", encoding="utf-8")
tunnel_process = subprocess.Popen(
    [str(CLOUDFLARED), "tunnel", "--url", "http://localhost:2333", "--no-autoupdate"],
    stdout=tunnel_log_handle,
    stderr=subprocess.STDOUT,
)

public_url = None
deadline = time.time() + 90
while time.time() < deadline:
    if tunnel_process.poll() is not None:
        print(TUNNEL_LOG.read_text(encoding="utf-8", errors="replace"))
        raise RuntimeError("cloudflared exited; log above.")
    match = re.search(
        r"https://[a-z0-9-]+\.trycloudflare\.com",
        TUNNEL_LOG.read_text(encoding="utf-8", errors="replace"),
    )
    if match:
        public_url = match.group(0)
        break
    time.sleep(3)

if public_url is None:
    raise TimeoutError(f"Tunnel URL not found; inspect {TUNNEL_LOG}.")

print("\nARDY interactive demo is public at:")
print(public_url)
print("\nOpen it in your browser. Both processes keep running in the background;")
print("on T4 (no TensorRT) the stream runs slower than real time - expected.")


---

## Troubleshooting

- **Only one GPU appears:** select `GPU T4 x2`, save, and restart the Kaggle session.
- **401/403 from Hugging Face:** accept the Llama license, wait for approval, and recreate `HF_TOKEN` with read access.
- **GPU 1 OOM while loading LLM2Vec:** restart the session to clear VRAM. If it persists, the next fallback is CPU text encoding or an explicit 8-bit/4-bit encoder implementation.
- **Package install fails:** restart the session after installation if Kaggle had already imported incompatible versions of `transformers` or `gradio`.
- **Text encoder exits:** inspect `/kaggle/working/ardy_text_encoder.log`. `ImportError: Found an incompatible version of torchao` means Kaggle's preinstalled torchao is still present -- rerun the install cell (it uninstalls torchao). `CUDA out of memory` means Llama-3-8B does not fit on one T4; the server cell detects this and retries the encoder on CPU automatically.
- **`unable to allocate shared memory (shm)`:** LLM2Vec switches to multi-process encoding when it sees 2+ GPUs, which Kaggle's small /dev/shm cannot handle. The server cell restricts the encoder to one visible device (`CUDA_VISIBLE_DEVICES=1`), forcing the single-process path.
- **Generation OOM on GPU 0:** keep `core8`, reduce `HISTORY_FRAMES` to `4`, shorten duration, and generate one sample at a time.
- **Interactive UI:** Kaggle does not expose container ports to your browser. Use the optional Cloudflare tunnel section (section 10) to open the Viser demo at a public HTTPS URL.

---

## Optional cleanup

In [ ]:
# Run this only when you are finished with the encoder service.
# text_encoder_process.terminate()
# text_encoder_process.wait(timeout=30)
# print("Text encoder service stopped.")
# If you also ran the interactive demo (section 10), stop it too:
# demo_process.terminate()
# tunnel_process.terminate()
# print('Demo and tunnel stopped.')


---

<div align='center'>

### 🎉 Enjoyed this notebook?

If this was helpful, please **upvote** and subscribe for more free AI tools!

<p align="center">
  <a href="https://www.youtube.com/@thebuildai?sub_confirmation=1"><img src="https://img.shields.io/badge/YouTube-SUBSCRIBE-red?style=for-the-badge&logo=youtube&logoColor=white" /></a>
  <a href="https://www.instagram.com/thebuildai/"><img src="https://img.shields.io/badge/Instagram-FOLLOW-E4405F?style=for-the-badge&logo=instagram&logoColor=white" /></a>
  <a href="https://www.tiktok.com/@the.build.ai"><img src="https://img.shields.io/badge/TikTok-FOLLOW-000000?style=for-the-badge&logo=tiktok&logoColor=white" /></a>
  <a href="https://github.com/cafermutluozkan"><img src="https://img.shields.io/badge/GitHub-FOLLOW-181717?style=for-the-badge&logo=github&logoColor=white" /></a>
</p>

  <p style='color: #888; margin-top: 15px;'>🏃 ARDY Motion Generator — Built by <strong><a href='https://www.thebuildai.tech/'>TheBuildAI</a></strong> 🌍</p>

</div>